In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import random

# Set seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Configuration - TỐI ƯU HÓA
class Config:
    model_name = "vinai/phobert-base"
    max_len = 128  # Giảm từ 256 xuống 128 để nhanh hơn
    batch_size = 32  # Tăng batch size
    epochs = 3  # Giảm epochs
    learning_rate = 2e-5
    weight_decay = 0.01
    n_folds = 3  # Giảm từ 5 xuống 3 folds
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_classes = 3
    num_topics = 4
    warmup_ratio = 0.1
    
config = Config()
print(f"Using device: {config.device}")

# Load data
train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sentiment distribution:\n{train_df['sentiment'].value_counts().sort_index()}")
print(f"Topic distribution:\n{train_df['topic'].value_counts().sort_index()}")

Using device: cuda
Train shape: (11322, 4)
Test shape: (4853, 2)
Sentiment distribution:
sentiment
0    5226
1     501
2    5595
Name: count, dtype: int64
Topic distribution:
topic
0    8132
1    2136
2     496
3     558
Name: count, dtype: int64


In [2]:
# Dataset class
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, has_labels=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        # Add topic
        if 'topic' in self.df.columns:
            item['topic'] = torch.tensor(self.df.iloc[idx]['topic'], dtype=torch.long)
        
        if self.has_labels:
            item['sentiment'] = torch.tensor(self.df.iloc[idx]['sentiment'], dtype=torch.long)
            
        return item

In [3]:
# Model với topic embedding
class PhoBERTWithTopic(nn.Module):
    def __init__(self, model_name, num_classes, num_topics):
        super(PhoBERTWithTopic, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.topic_embedding = nn.Embedding(num_topics, 64)
        self.dropout = nn.Dropout(0.3)
        
        hidden_size = self.bert.config.hidden_size + 64
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, input_ids, attention_mask, topic_ids):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = bert_outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        
        topic_emb = self.topic_embedding(topic_ids)
        combined = torch.cat([pooled, topic_emb], dim=1)
        
        logits = self.classifier(combined)
        return logits

In [4]:
# Training function
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        topic_ids = batch['topic'].to(device)
        labels = batch['sentiment'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, topic_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1

# Evaluation function
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            topic_ids = batch['topic'].to(device)
            labels = batch['sentiment'].to(device)
            
            outputs = model(input_ids, attention_mask, topic_ids)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1

# Prediction function
def predict(model, loader, device):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            topic_ids = batch['topic'].to(device)
            
            outputs = model(input_ids, attention_mask, topic_ids)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    return all_preds

In [5]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

# Prepare data - SỬ DỤNG TOPIC CÓ SẴN TRONG TRAIN
# Không cần train topic classifier riêng
print("\n" + "="*50)
print("Training Sentiment Classifier with Topic Information")
print("="*50)

# Cross-validation
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_predictions = []
val_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*40}")
    print(f"Fold {fold + 1}/{config.n_folds}")
    print(f"{'='*40}")
    
    # Split data
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Create datasets
    train_dataset = SentimentDataset(train_fold, tokenizer, config.max_len, has_labels=True)
    val_dataset = SentimentDataset(val_fold, tokenizer, config.max_len, has_labels=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)
    
    # Initialize model
    print("Initializing model...")
    model = PhoBERTWithTopic(config.model_name, config.num_classes, config.num_topics).to(config.device)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, criterion, config.device)
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, config.device)
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"✓ New best model! Val F1: {val_f1:.4f}")
    
    # Load best model
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    # Create test dataset with topics
    # For test set, we need to predict topics first using a simple approach
    # Use the mode topic from training data as fallback
    test_df_copy = test_df.copy()
    
    # For each test sample, find similar sentences in train to predict topic
    # Simple approach: use the most frequent topic
    most_common_topic = train_df['topic'].mode()[0]
    test_df_copy['topic'] = most_common_topic
    
    # Alternatively, you can train a simple topic classifier here
    # But for speed, we'll use the mode
    
    test_dataset = SentimentDataset(test_df_copy, tokenizer, config.max_len, has_labels=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)
    
    print("Predicting on test set...")
    fold_pred = predict(model, test_loader, config.device)
    fold_predictions.append(fold_pred)
    
    # Clear GPU memory
    del model
    torch.cuda.empty_cache()

print(f"\n{'='*50}")
print(f"Cross-validation F1 scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")

Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Train - Loss: 0.1884, Acc: 0.9463, F1: 0.9441


Evaluating: 100%|██████████| 118/118 [00:25<00:00,  4.59it/s]


Val   - Loss: 0.2278, Acc: 0.9322, F1: 0.9308
✓ New best model! Val F1: 0.9308
Predicting on test set...


Predicting: 100%|██████████| 152/152 [00:32<00:00,  4.62it/s]


Cross-validation F1 scores: [0.9310467820710211, 0.9260424990118263, 0.9307888367056232]
Mean CV F1: 0.9293 (+/- 0.0023)


In [6]:
# Ensemble predictions (majority voting)
fold_predictions = np.array(fold_predictions)
final_predictions = []

print("\n" + "="*50)
print("Ensembling predictions...")
print("="*50)

for i in range(fold_predictions.shape[1]):
    pred_counts = np.bincount(fold_predictions[:, i])
    final_predictions.append(np.argmax(pred_counts))

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("\n✓ Submission saved to submission.csv")
print("\nSentiment distribution in predictions:")
print(submission['sentiment'].value_counts().sort_index())

# Weighted ensemble
if len(val_scores) > 0:
    weights = np.array(val_scores) / np.sum(val_scores)
    weighted_predictions = []
    
    for i in range(fold_predictions.shape[1]):
        weighted_votes = np.zeros(config.num_classes)
        for fold_idx in range(len(fold_predictions)):
            weighted_votes[fold_predictions[fold_idx, i]] += weights[fold_idx]
        weighted_predictions.append(np.argmax(weighted_votes))
    
    submission_weighted = pd.DataFrame({
        'id': test_df['id'],
        'sentiment': weighted_predictions
    })
    submission_weighted.to_csv('submission_weighted.csv', index=False)
    print("\n✓ Weighted ensemble saved to submission_weighted.csv")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Number of folds: {config.n_folds}")
print(f"Epochs per fold: {config.epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Max sequence length: {config.max_len}")
print(f"Mean CV F1 Score: {np.mean(val_scores):.4f}")
print(f"Best fold F1: {max(val_scores):.4f}")
print("\n✓ Done!")


Ensembling predictions...

✓ Submission saved to submission.csv

Sentiment distribution in predictions:
sentiment
0    2252
1     109
2    2492
Name: count, dtype: int64

✓ Weighted ensemble saved to submission_weighted.csv

SUMMARY
Number of folds: 3
Epochs per fold: 3
Batch size: 32
Max sequence length: 128
Mean CV F1 Score: 0.9293
Best fold F1: 0.9310

✓ Done!


In [7]:
# Optional: Quick evaluation on validation set
print("\n" + "="*50)
print("Quick evaluation on training data")
print("="*50)

# Train final model on full data (optional)
print("Training final model on full dataset...")
full_dataset = SentimentDataset(train_df, tokenizer, config.max_len, has_labels=True)
full_loader = DataLoader(full_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)

final_model = PhoBERTWithTopic(config.model_name, config.num_classes, config.num_topics).to(config.device)
optimizer = AdamW(final_model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = len(full_loader) * 2  # 2 epochs for final model
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.CrossEntropyLoss()

for epoch in range(2):
    train_loss, train_acc, train_f1 = train_epoch(final_model, full_loader, optimizer, scheduler, criterion, config.device)
    print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Acc={train_acc:.4f}, F1={train_f1:.4f}")

# Save final model
torch.save(final_model.state_dict(), 'final_model.pth')
print("\n✓ Final model saved to final_model.pth")


Quick evaluation on training data
Training final model on full dataset...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Training: 100%|██████████| 354/354 [04:11<00:00,  1.41it/s]


Epoch 1: Loss=0.4451, Acc=0.8456, F1=0.8295


Training: 100%|██████████| 354/354 [04:10<00:00,  1.41it/s]


Epoch 2: Loss=0.2329, Acc=0.9302, F1=0.9250

✓ Final model saved to final_model.pth
